# ESMCA v8 — LLaMA-3.2 3B + MultiNLI

The original ESMCA idea on the right backbone:
> *Attribution maps route LoRA adapters without task labels and detect forgetting — one signal, two jobs.*

v1–v7 failed because RoBERTa (125M frozen) produces degenerate attribution maps.
LLaMA-3.2 3B (3B params, 15T token pretraining) produces rich semantic representations
where genre-specific vocabulary activates genuinely different patterns.

**Before running:**
1. Accept license at huggingface.co/meta-llama/Llama-3.2-3B
2. Kaggle → Settings → Secrets → add `HF_TOKEN`

In [2]:
!pip install -q "captum>=0.7.0" "datasets>=2.19.0" "bitsandbytes>=0.43.0" "accelerate>=0.27.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 6.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.8 MB/s eta 0:00:00:00:0100:01


In [3]:
import os, json, random, logging, collections, time, warnings
from dataclasses import dataclass
from typing import Dict, List, Optional
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from captum.attr import LayerIntegratedGradients
from scipy.stats import spearmanr
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, confusion_matrix
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from huggingface_hub.utils import logging as hf_logging
from datasets.utils import logging as ds_logging
from transformers.utils import logging as tf_logging
hf_logging.set_verbosity_error(); ds_logging.set_verbosity_error(); tf_logging.set_verbosity_error()
logging.getLogger("httpx").setLevel(logging.WARNING)
warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger("esmca")
NUM_CLASSES = 3

In [4]:
import os
HF_TOKEN = os.environ.get("HF_TOKEN", None)
if HF_TOKEN is None:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded from Kaggle secrets.")
    except Exception:
        print("WARNING: HF_TOKEN not found. Add it in Kaggle Secrets.")

CONFIG = {
    "backbone":            "meta-llama/Llama-3.2-3B",
    "max_seq_len":         128,
    "lora_rank":           8,
    "lora_alpha":          16,
    "genres": ["fiction", "government", "slate", "telephone", "travel"],
    "train_per_genre":     200,
    "batch_size":          4,
    "epochs_per_task":     5,
    "lr":                  2e-4,
    "n_prototype_samples": 64,
    "ig_steps":            8,
    "router_tau":          0.1,
    "drift_delta":         0.05,
    "attrib_loss_weight":  1.0,
    "seed":                42,
    "output_dir": "/kaggle/working" if os.path.isdir("/kaggle/working") else "outputs",
}

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
os.makedirs(CONFIG["output_dir"], exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

HF_TOKEN loaded from Kaggle secrets.
Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 2. Load LLaMA-3.2 3B (4-bit NF4)

Memory: model ~2.5GB + LoRA ~0.3GB + heads ~0.1GB + activations/IG ~5GB ≈ 8GB total — fits T4 comfortably.

In [5]:
def load_backbone(model_name, hf_token=None):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    backbone = AutoModel.from_pretrained(
        model_name, quantization_config=bnb_config,
        device_map={"": 0}, torch_dtype=torch.bfloat16,
        token=hf_token, trust_remote_code=True,
    )
    for param in backbone.parameters():
        param.requires_grad = False
    backbone.eval()
    torch.cuda.empty_cache()
    used  = torch.cuda.memory_allocated()/1e9
    total = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f"Backbone loaded. VRAM: {used:.1f}/{total:.1f} GB")
    return backbone, tokenizer


def mean_pool(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    return (hidden_states * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)


class ClassificationHead(nn.Module):
    def __init__(self, hidden, n_classes=NUM_CLASSES):
        super().__init__()
        self.dense    = nn.Linear(hidden, hidden)
        self.dropout  = nn.Dropout(0.1)
        self.out_proj = nn.Linear(hidden, n_classes)
        self.act      = nn.Tanh()
    def forward(self, x):
        return self.out_proj(self.dropout(self.act(self.dense(x))))

## 3. LoRA adapter bank (LLaMA targets: q_proj + v_proj)

In [6]:
class LoRAPair(nn.Module):
    def __init__(self, d_in, d_out, rank):
        super().__init__()
        self.A = nn.Parameter(torch.empty(rank, d_in, dtype=torch.bfloat16))
        self.B = nn.Parameter(torch.zeros(d_out, rank, dtype=torch.bfloat16))
        nn.init.kaiming_uniform_(self.A, a=5**0.5)
    def forward(self, x): return (x @ self.A.t()) @ self.B.t()


class LoRAInjectedLinear(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        self.scaling = alpha / rank
        self.rank = rank
        self.in_features  = base.in_features
        self.out_features = base.out_features
        self.adapters        = nn.ModuleDict()
        self.active_task     = None
        self.routing_weights = None

    def add_task(self, name):
        # Create adapter
        adapter = LoRAPair(self.in_features, self.out_features, self.rank)
        # FIX: Move adapter to the device of the base layer
        device = next(self.base.parameters()).device
        adapter = adapter.to(device)
        self.adapters[name] = adapter
    def freeze_task(self, name):
        for p in self.adapters[name].parameters(): p.requires_grad = False
    def set_active_task(self, name):   self.active_task, self.routing_weights = name, None
    def set_routing_weights(self, w):  self.routing_weights, self.active_task = w, None

    def forward(self, x):
        base_device = self.base.weight.device
        if x.device != base_device:
            x = x.to(base_device)
        out = self.base(x)
        if self.active_task and self.active_task in self.adapters:
            adapter = self.adapters[self.active_task]
            if next(adapter.parameters()).device != base_device:
                adapter.to(base_device)
            out = out + self.scaling * adapter(x)
        elif self.routing_weights:
            for name, w in self.routing_weights.items():
                if w and name in self.adapters:
                    adapter = self.adapters[name]
                    if next(adapter.parameters()).device != base_device:
                        adapter.to(base_device)
                    out = out + w * self.scaling * adapter(x)
        return out


class AdapterBank:
    def __init__(self, injected):
        self.injected, self.task_order = injected, []
    def add_task(self, name):
        self.task_order.append(name)
        for _, m in self.injected: m.add_task(name)
    def freeze_task(self, name):
        for _, m in self.injected: m.freeze_task(name)
    def set_active_task(self, name):
        for _, m in self.injected: m.set_active_task(name)
    def set_routing_weights(self, w):
        for _, m in self.injected: m.set_routing_weights(w)
    def trainable_parameters(self, name):
        for _, m in self.injected: yield from m.adapters[name].parameters()


def inject_lora(backbone, rank, alpha):
    injected = []
    # CHANGE THIS: backbone.model.layers -> backbone.layers
    for li, layer in enumerate(backbone.layers):
        for attr in ("q_proj", "v_proj"):
            base = getattr(layer.self_attn, attr)
            wrapped = LoRAInjectedLinear(base, rank, alpha)
            setattr(layer.self_attn, attr, wrapped)
            injected.append((f"layer{li}.{attr}", wrapped))
    logger.info(f"LoRA injected: {len(injected)} projections ({len(backbone.layers)} layers x 2)")
    return AdapterBank(injected)

## 4. ESMCAModel (LLaMA — mean pooling, embed_tokens, per-task heads)

In [7]:
class ESMCAModel(nn.Module):
    def __init__(self, backbone_name, lora_rank, lora_alpha, hf_token=None):
        super().__init__()
        self.backbone, self.tokenizer = load_backbone(backbone_name, hf_token)
        self.adapter_bank = inject_lora(self.backbone, lora_rank, lora_alpha)
        self.heads = nn.ModuleDict()
        self.probe_head: Optional[ClassificationHead] = None
        self._hidden = self.backbone.config.hidden_size

    def _pool(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        return mean_pool(out.last_hidden_state, attention_mask)

    def forward_task(self, input_ids, attention_mask, task):
        self.adapter_bank.set_active_task(task)
        return self.heads[task](self._pool(input_ids, attention_mask))

    def forward_composed(self, input_ids, attention_mask, weights):
        self.adapter_bank.set_routing_weights(weights)
        pooled = self._pool(input_ids, attention_mask)
        self.adapter_bank.set_routing_weights(None)
        logits = None
        for t, w in weights.items():
            if w and t in self.heads:
                l = self.heads[t](pooled) * w
                logits = l if logits is None else logits + l
        return logits

    def forward_frozen(self, input_ids, attention_mask):
        self.adapter_bank.set_active_task(None)
        return self.probe_head(self._pool(input_ids, attention_mask))

    def start_new_task(self, name):
        self.adapter_bank.add_task(name)
        # Determine active device
        dev = next(p for p in self.backbone.parameters() if p.device.type == "cuda").device
        
        # Instantiate heads on GPU
        self.heads[name] = ClassificationHead(self._hidden).to(dev)
        if self.probe_head is None:
            self.probe_head = ClassificationHead(self._hidden).to(dev)
            
        # FIX: Ensure all adapter parameters are explicitly on the GPU
        for _, m in self.adapter_bank.injected:
            if name in m.adapters:
                m.adapters[name] = m.adapters[name].to(dev)
                
        self.adapter_bank.set_active_task(name)

    def freeze_task(self, name):
        self.adapter_bank.freeze_task(name)
        for p in self.heads[name].parameters(): p.requires_grad = False

    def task_trainable_parameters(self, name):
        yield from self.adapter_bank.trainable_parameters(name)
        yield from self.heads[name].parameters()

## 5. AME — Attribution Map Extractor

Anchored to `model.embed_tokens` (LLaMA's embedding layer, not RoBERTa's `backbone.embeddings`).
Single-pass frozen backbone IG — the original clean ESMCA design.

In [8]:
class AttributionMapExtractor:
    def __init__(self, model: ESMCAModel, n_steps: int = 20):
        self.model, self.n_steps = model, n_steps
        self._ig = LayerIntegratedGradients(
            self._frozen_fwd,
            model.backbone.embed_tokens   # FIXED: removed extra .model
        )

    def _frozen_fwd(self, input_ids, attention_mask):
        return self.model.forward_frozen(input_ids, attention_mask)

    def _token_scores(self, input_ids, attention_mask):
        self.model.eval()
        with torch.no_grad():
            target = self.model.forward_frozen(input_ids, attention_mask).argmax(-1)
        pad = self.model.tokenizer.pad_token_id
        with torch.enable_grad():
            attrs = self._ig.attribute(
                inputs=input_ids,
                baselines=torch.full_like(input_ids, pad),
                additional_forward_args=(attention_mask,),
                target=target,
                n_steps=self.n_steps,
                internal_batch_size=input_ids.shape[0],  # cap peak mem: process IG steps in chunks of batch_size
            ).sum(dim=-1).detach()
        return attrs * attention_mask

    def attribute(self, input_ids, attention_mask):
        scores = self._token_scores(input_ids, attention_mask).abs()
        with torch.no_grad():
            E = self.model.backbone.embed_tokens(input_ids).float() # FIXED
        w = scores / (scores.sum(dim=1, keepdim=True) + 1e-8)
        s = torch.einsum("bl,bld->bd", w, E)
        return F.normalize(s, dim=-1)

    def attribute_differentiable(self, input_ids, attention_mask, target):
        embed_fn = self.model.backbone.embed_tokens # FIXED
        embeds = embed_fn(input_ids).detach().float().requires_grad_(True)
        out = self.model.backbone(inputs_embeds=embeds.to(torch.bfloat16),
                                  attention_mask=attention_mask)
        pooled = mean_pool(out.last_hidden_state.float(), attention_mask)
        task = self.model.adapter_bank.injected[0][1].active_task
        head = self.model.heads[task] if task in self.model.heads else self.model.probe_head
        logits = head(pooled)
        selected = logits.gather(1, target.unsqueeze(1)).squeeze(1).sum()
        (grads,) = torch.autograd.grad(selected, embeds, create_graph=True)
        scores = ((embeds * grads).sum(dim=-1) * attention_mask).abs()
        with torch.no_grad():
            E = embed_fn(input_ids).float()
        w = scores / (scores.sum(dim=1, keepdim=True) + 1e-8)
        return F.normalize(torch.einsum("bl,bld->bd", w, E), dim=-1)

    def compute_prototype(self, loader, n_samples=256):
        dev = next(p for p in self.model.backbone.parameters() if p.device.type=="cuda").device
        collected, seen = [], 0
        for batch in loader:
            if seen >= n_samples: break
            batch = {k: v.to(dev) for k, v in batch.items()}
            a = self.attribute(batch["input_ids"], batch["attention_mask"])
            collected.append(a.cpu()); seen += a.shape[0]
        return F.normalize(torch.cat(collected)[:n_samples].mean(0), dim=-1)

## 6. Router + Drift Monitor — the original dual-use design

In [9]:
class AttributionBasedRouter:
    def __init__(self, tau=0.1): self.tau = tau
    def route(self, s_x, prototypes):
        names = list(prototypes.keys())
        sims = torch.stack([F.cosine_similarity(s_x.unsqueeze(0), prototypes[t].unsqueeze(0)).squeeze(0)
                            for t in names])
        w = F.softmax(sims / self.tau, dim=0)
        return {t: float(x) for t, x in zip(names, w)}


class ExplanationDriftMonitor:
    def __init__(self, delta=0.05, lam=1.0):
        self.delta, self.lam = delta, lam
    def drift(self, current, prototype):
        return float(1.0 - F.cosine_similarity(current.unsqueeze(0), prototype.unsqueeze(0)).squeeze(0))
    def check_all(self, current, prototypes):
        return {t: self.drift(current[t], prototypes[t]) for t in prototypes if t in current}
    def flagged(self, scores): return [t for t, d in scores.items() if d > self.delta]
    def attribution_loss(self, current, prototype):
        return self.lam * torch.sum((current - prototype.to(current.device))**2)

## 7. MultiNLI — 5 matched genres as continual tasks

Premise + [SEP] + hypothesis. 3-way NLI (entailment/neutral/contradiction).
Filter label==-1. val_matched for proto/eval; val_mismatched for cross-genre generalization.

In [10]:
@dataclass
class Task:
    name: str
    train_loader: DataLoader
    proto_loader: DataLoader
    eval_loader:  DataLoader
    mismatch_loader: Optional[DataLoader] = None


class MultiNLIDataset(Dataset):
    def __init__(self, examples, tokenizer, max_seq_len):
        self.examples, self.tok, self.L = examples, tokenizer, max_seq_len
    def __len__(self): return len(self.examples)
    def __getitem__(self, i):
        ex  = self.examples[i]
        txt = ex["premise"] + " [SEP] " + ex["hypothesis"]
        enc = self.tok(txt, truncation=True, max_length=self.L,
                       padding="max_length", return_tensors="pt")
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(ex["label"], dtype=torch.long),
        }


def load_multinli_tasks(tokenizer, max_seq_len, batch_size,
                        genres=None, train_per_genre=200, seed=42):
    if genres is None:
        genres = ["fiction","government","slate","telephone","travel"]
    raw = load_dataset("nyu-mll/multi_nli")
    rng = np.random.RandomState(seed)
    train_all = raw["train"].filter(lambda x: x["label"] != -1)
    val_m     = raw["validation_matched"].filter(lambda x: x["label"] != -1)
    val_mm    = raw["validation_mismatched"].filter(lambda x: x["label"] != -1)
    tasks = []
    for genre in genres:
        tr = [ex for ex in train_all if ex["genre"] == genre]
        per_label = train_per_genre // 3
        tr_ex = []
        for lbl in range(3):
            sub = [ex for ex in tr if ex["label"] == lbl]
            chosen = rng.permutation(len(sub))[:per_label]
            tr_ex += [sub[i] for i in chosen]
        rng.shuffle(tr_ex)
        va  = [ex for ex in val_m  if ex["genre"] == genre]
        mm  = [ex for ex in val_mm if ex["genre"] == genre]
        cut = len(va)//2
        proto_ex, eval_ex = va[:cut], va[cut:]
        mk = lambda exs, sh: DataLoader(MultiNLIDataset(exs, tokenizer, max_seq_len),
                                        batch_size=batch_size, shuffle=sh, drop_last=False)
        tasks.append(Task(genre, mk(tr_ex,True), mk(proto_ex,False),
                          mk(eval_ex,False), mk(mm,False) if mm else None))
        logger.info(f"{genre:12s} train={len(tr_ex)} proto={len(proto_ex)} eval={len(eval_ex)} mismatch={len(mm)}")
    return tasks

## 8. ContinualTrainer

In [11]:
class ContinualTrainer:
    def __init__(self, model, cfg):
        self.model  = model
        self.cfg    = cfg
        self.ame    = AttributionMapExtractor(model, n_steps=cfg["ig_steps"])
        self.router = AttributionBasedRouter(tau=cfg["router_tau"])
        self.edm    = ExplanationDriftMonitor(delta=cfg["drift_delta"], lam=cfg["attrib_loss_weight"])
        self.prototypes: Dict[str, torch.Tensor] = {}
        self.tasks:      Dict[str, Task]         = {}
        self.drift_history: Dict[str, Dict[str, float]] = {}
        self.device = next(p for p in model.backbone.parameters() if p.device.type=="cuda").device

    def _to(self, b): return {k: v.to(self.device) for k, v in b.items()}

    def _pre_training_drift_check(self, incoming):
        if not self.prototypes: return []
        current = {}
        for name, task in self.tasks.items():
            b = self._to(next(iter(task.proto_loader)))
            current[name] = self.ame.attribute(b["input_ids"], b["attention_mask"]).mean(0).cpu()
        scores = self.edm.check_all(current, {k: v.cpu() for k, v in self.prototypes.items()})
        self.drift_history[incoming] = scores
        fl = self.edm.flagged(scores)
        if fl: logger.info(f"EDM drift flagged: {fl}")
        return fl

    def _consolidation_step(self, optimizer, flagged, task_name):
        if not flagged or not self.prototypes: return
        optimizer.zero_grad()
        total = torch.tensor(0.0, device=self.device)
        self.model.adapter_bank.set_active_task(task_name)
        for past in flagged:
            if past not in self.prototypes: continue
            b = self._to(next(iter(self.tasks[past].proto_loader)))
            with torch.no_grad():
                tgt = self.model.forward_frozen(b["input_ids"], b["attention_mask"]).argmax(-1)
            self.model.adapter_bank.set_active_task(task_name)
            a = self.ame.attribute_differentiable(b["input_ids"], b["attention_mask"], tgt)
            total = total + self.edm.attribution_loss(a.mean(0), self.prototypes[past])
        if total.item() > 0:
            total.backward(); optimizer.step()

    def train_task(self, task):
        name = task.name
        self.tasks[name] = task
        flagged = self._pre_training_drift_check(name)
        self.model.start_new_task(name)
        torch.cuda.empty_cache()
        optimizer = AdamW(list(self.model.task_trainable_parameters(name)), lr=self.cfg["lr"])
        for ep in range(self.cfg["epochs_per_task"]):
            self.model.backbone.eval()
            for h in self.model.heads.values(): h.train()
            if self.model.probe_head: self.model.probe_head.train()
            bar = tqdm(task.train_loader,
                       desc=f"[{name}] ep {ep+1}/{self.cfg['epochs_per_task']}", leave=False)
            for b in bar:
                b = self._to(b)
                optimizer.zero_grad()
                loss = F.cross_entropy(
                    self.model.forward_task(b["input_ids"], b["attention_mask"], name),
                    b["labels"])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    list(self.model.task_trainable_parameters(name)), 1.0)
                optimizer.step()
                bar.set_postfix(loss=f"{loss.item():.3f}")
            self._consolidation_step(optimizer, flagged, name)
        self.model.backbone.eval()
        torch.cuda.empty_cache()
        proto = self.ame.compute_prototype(task.proto_loader, n_samples=self.cfg["n_prototype_samples"])
        self.prototypes[name] = proto.cpu()
        self.model.freeze_task(name)
        if len(self.prototypes) > 1:
            names = list(self.prototypes.keys())
            P = torch.stack([self.prototypes[n] for n in names])
            S = (P @ P.t()).numpy()
            off = S[~np.eye(len(names), dtype=bool)]
            logger.info(f"Done '{name}'. Proto off-diag: mean={off.mean():.3f} max={off.max():.3f}")
        else:
            logger.info(f"Done '{name}'.")

## 9. Three inference modes

In [12]:
@torch.no_grad()
def eval_oracle(trainer, task):
    trainer.model.backbone.eval()
    correct = total = 0
    for b in task.eval_loader:
        b = trainer._to(b)
        preds = trainer.model.forward_task(b["input_ids"], b["attention_mask"], task.name).argmax(-1)
        correct += (preds == b["labels"]).sum().item(); total += len(preds)
    return correct/total if total else 0.0


@torch.no_grad()
def eval_uniform(trainer, task):
    trainer.model.backbone.eval()
    names = list(trainer.prototypes.keys())
    w = {t: 1.0/len(names) for t in names}
    correct = total = 0
    for b in task.eval_loader:
        b = trainer._to(b)
        preds = trainer.model.forward_composed(b["input_ids"], b["attention_mask"], w).argmax(-1)
        correct += (preds == b["labels"]).sum().item(); total += len(preds)
    return correct/total if total else 0.0


@torch.no_grad()
def eval_esmca(trainer, task, true_task_idx, task_order, loader=None):
    trainer.model.backbone.eval()
    if loader is None: loader = task.eval_loader
    dev    = trainer.device
    protos = {k: v.to(dev) for k, v in trainer.prototypes.items()}
    correct = total = 0
    routed_idx, true_idx = [], []
    for b in tqdm(loader, desc=f"eval {task.name}", leave=False):
        b = trainer._to(b)
        s = trainer.ame.attribute(b["input_ids"], b["attention_mask"])
        for i in range(s.shape[0]):
            w = trainer.router.route(s[i], protos)
            routed_idx.append(task_order.index(max(w, key=w.get)))
            true_idx.append(true_task_idx)
            logits = trainer.model.forward_composed(
                b["input_ids"][i:i+1], b["attention_mask"][i:i+1], w)
            correct += int(logits.argmax(-1).item() == b["labels"][i].item())
            total   += 1
    return (correct/total if total else 0.0), routed_idx, true_idx

## 10. Metrics + separability

In [13]:
def compute_acc(R): return float(np.mean(R[-1]))
def compute_bwt(R):
    T=len(R); return 0.0 if T<2 else float(np.mean([R[T-1][t]-R[t][t] for t in range(T-1)]))
def compute_fwt(R):
    T=len(R); return 0.0 if T<2 else float(np.mean([R[i-1][i] for i in range(1,T)]))
def attribution_drift_score(hist):
    v=[d for per in hist.values() for d in per.values()]; return float(np.mean(v)) if v else 0.0
def ris(true_idx, routed_idx):
    if len(true_idx)<2: return 0.0
    c,_=spearmanr(true_idx,routed_idx); return float(c) if c==c else 0.0

@torch.no_grad()
def collect_sample_attributions(trainer, tasks, per_task=40):
    X, y, names = [], [], [t.name for t in tasks]
    for j, task in enumerate(tasks):
        seen = 0
        for b in task.eval_loader:
            if seen >= per_task: break
            b = trainer._to(b)
            a = trainer.ame.attribute(b["input_ids"], b["attention_mask"])
            take = min(per_task-seen, a.shape[0])
            X.append(a[:take].cpu()); y += [j]*take; seen += take
    return torch.cat(X).numpy(), np.array(y), names

def separability_report(X, y, names, out_dir):
    sil  = float(silhouette_score(X, y)) if len(names)>1 else 0.0
    perp = max(2, min(30, len(X)//4))
    emb  = TSNE(n_components=2, perplexity=perp, init="pca", random_state=0).fit_transform(X)
    plt.figure(figsize=(7,6))
    for j,n in enumerate(names):
        m=y==j; plt.scatter(emb[m,0],emb[m,1],s=18,label=n,alpha=0.8)
    plt.legend(fontsize=9)
    plt.title(f"IG attribution embeddings — LLaMA 3.2 3B (silhouette={sil:.3f})")
    p = os.path.join(out_dir, "attribution_tsne_v8.png")
    plt.tight_layout(); plt.savefig(p, dpi=150); plt.close()
    print(f"t-SNE saved -> {p}")
    return sil

## 11. Sanity gate

Check (c) is the decisive one: if fiction vs government IG cosine < 0.95 before training,
LLaMA's representations are discriminative enough for routing to work.

In [14]:
model   = ESMCAModel(CONFIG["backbone"], CONFIG["lora_rank"], CONFIG["lora_alpha"], hf_token=HF_TOKEN)
tasks   = load_multinli_tasks(model.tokenizer, CONFIG["max_seq_len"], CONFIG["batch_size"],
                               CONFIG["genres"], CONFIG["train_per_genre"], CONFIG["seed"])
trainer = ContinualTrainer(model, CONFIG)

print(f"\nVRAM after load: {torch.cuda.memory_allocated()/1e9:.1f} / "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

_b = trainer._to(next(iter(tasks[0].train_loader)))
trainer.model.start_new_task("_probe")
torch.cuda.empty_cache()
_opt = AdamW(list(trainer.model.task_trainable_parameters("_probe")), lr=CONFIG["lr"])
trainer.model.backbone.eval(); trainer.model.heads["_probe"].train()
for step in range(40):
    _opt.zero_grad()
    _loss = F.cross_entropy(
        trainer.model.forward_task(_b["input_ids"], _b["attention_mask"], "_probe"), _b["labels"])
    _loss.backward()
    torch.nn.utils.clip_grad_norm_(list(trainer.model.task_trainable_parameters("_probe")), 1.0)
    _opt.step()
trainer.model.backbone.eval()
with torch.no_grad():
    _on  = trainer.model.forward_task(_b["input_ids"], _b["attention_mask"], "_probe")
    trainer.model.adapter_bank.set_active_task(None)
    _off = trainer.model.probe_head(
        mean_pool(trainer.model.backbone(_b["input_ids"], _b["attention_mask"]).last_hidden_state.float(),
                  _b["attention_mask"]))
chance = float(np.log(NUM_CLASSES))
print(f"(a) probe loss after 40 steps: {_loss.item():.3f}  (chance={chance:.3f})")
print(f"(b) adapter max|delta logit|: {(_on.float()-_off.float()).abs().max().item():.3f}")
assert _loss.item() < 0.7 * chance, "SANITY FAIL (a): loss stuck — check LR or HF_TOKEN"
assert (_on.float()-_off.float()).abs().max().item() > 0.1, "SANITY FAIL (b): adapters inert"

with torch.no_grad():
    b0 = trainer._to(next(iter(tasks[0].eval_loader)))
    b1 = trainer._to(next(iter(tasks[1].eval_loader)))
    s0 = trainer.ame.attribute(b0["input_ids"], b0["attention_mask"]).mean(0)
    s1 = trainer.ame.attribute(b1["input_ids"], b1["attention_mask"]).mean(0)
    cos = F.cosine_similarity(s0.unsqueeze(0), s1.unsqueeze(0)).item()
print(f"(c) fiction vs government IG cosine: {cos:.3f}")
print(f"    (RoBERTa gave 1.000 -- anything below 0.98 is better)")
if cos < 0.95:   print("    Strong separation! Routing should work.")
elif cos < 0.99: print("    Marginal -- better than RoBERTa, worth running.")
else:            print("    WARNING: still near 1.0 -- check ig_steps or embed layer path")
print("\nSanity gate PASSED -- safe to run full training.")
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

Filter:   0%|          | 0/392702 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9815 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9832 [00:00<?, ? examples/s]

fiction      train=198 proto=986 eval=987 mismatch=0
government   train=198 proto=972 eval=973 mismatch=0
slate        train=198 proto=977 eval=978 mismatch=0
telephone    train=198 proto=983 eval=983 mismatch=0
travel       train=198 proto=988 eval=988 mismatch=0



VRAM after load: 2.2 / 15.6 GB
(a) probe loss after 40 steps: 0.000  (chance=1.099)
(b) adapter max|delta logit|: 14.145
(c) fiction vs government IG cosine: 0.984
    (RoBERTa gave 1.000 -- anything below 0.98 is better)
    Marginal -- better than RoBERTa, worth running.

Sanity gate PASSED -- safe to run full training.


## 12. Sequential training

In [15]:
t_start = time.time()
for i, task in enumerate(tasks):
    trainer.train_task(task)
    elapsed   = (time.time()-t_start)/60
    remaining = elapsed/(i+1)*(len(tasks)-i-1)
    logger.info(f"[{i+1}/{len(tasks)}] elapsed {elapsed:.1f} min | est. remaining ~{remaining:.0f} min")
    torch.cuda.empty_cache()
print(f"\nPhase 1 complete in {(time.time()-t_start)/60:.1f} min.")

[fiction] ep 1/5:   0%|          | 0/50 [00:00<?, ?it/s]

[fiction] ep 2/5:   0%|          | 0/50 [00:00<?, ?it/s]

[fiction] ep 3/5:   0%|          | 0/50 [00:00<?, ?it/s]

[fiction] ep 4/5:   0%|          | 0/50 [00:00<?, ?it/s]

[fiction] ep 5/5:   0%|          | 0/50 [00:00<?, ?it/s]

...

[travel] ep 3/5:   0%|          | 0/50 [00:00<?, ?it/s]

[travel] ep 4/5:   0%|          | 0/50 [00:00<?, ?it/s]

[travel] ep 5/5:   0%|          | 0/50 [00:00<?, ?it/s]

Done 'travel'. Proto off-diag: mean=0.994 max=0.999
[5/5] elapsed 98.7 min | est. remaining ~0 min



Phase 1 complete in 98.7 min.


## 13. R-matrix

In [ ]:
task_order = trainer.model.adapter_bank.task_order
R = []; row_times = []; t_start = time.time()
for i, task in enumerate(tasks):
    t_row = time.time(); row = []
    for j, et in enumerate(tasks):
        t0  = time.time()
        acc, _, _ = eval_esmca(trainer, et, j, task_order)
        row.append(acc)
        print(f"  eval {et.name:12s} acc={acc:.3f}  ({time.time()-t0:.0f}s)")
    R.append(row); row_times.append(time.time()-t_row)
    left = (len(tasks)-i-1)*np.mean(row_times)/60
    print(f"[{i+1}/{len(tasks)}] '{task.name}' | elapsed {(time.time()-t_start)/60:.1f} min | ~{left:.0f} min left")
    torch.cuda.empty_cache()

eval fiction:   0%|          | 0/247 [00:00<?, ?it/s]

  eval fiction      acc=0.379  (6794s)


eval government:   0%|          | 0/244 [00:00<?, ?it/s]

  eval government   acc=0.367  (6771s)


eval slate:   0%|          | 0/245 [00:00<?, ?it/s]

...

eval telephone:   0%|          | 0/246 [00:00<?, ?it/s]

  eval telephone    acc=0.364  (6823s)


eval travel:   0%|          | 0/247 [00:00<?, ?it/s]

  eval travel       acc=0.370  (6896s)
[1/5] 'fiction' | elapsed 567.9 min | ~2272 min left


eval fiction:   0%|          | 0/247 [00:00<?, ?it/s]

## 14. Final three-way comparison + cross-genre generalization

In [ ]:
oracle_accs, uniform_accs, esmca_accs = [], [], []
all_true, all_routed = [], []
for j, task in enumerate(tasks):
    o = eval_oracle(trainer, task)
    u = eval_uniform(trainer, task)
    e, r_idx, t_idx = eval_esmca(trainer, task, j, task_order)
    oracle_accs.append(o); uniform_accs.append(u); esmca_accs.append(e)
    all_routed += r_idx; all_true += t_idx
    print(f"{task.name:12s}  oracle={o:.3f}  ESMCA={e:.3f}  uniform={u:.3f}")

cm     = confusion_matrix(all_true, all_routed, labels=list(range(len(tasks))))
recall = np.diag(cm).astype(float)/(cm.sum(axis=1)+1e-8)
balanced_routing = float(np.mean(recall))
routing_acc = float(np.mean(np.array(all_true)==np.array(all_routed)))
print(f"\nMean:  oracle={np.mean(oracle_accs):.3f}  ESMCA={np.mean(esmca_accs):.3f}  uniform={np.mean(uniform_accs):.3f}")
print(f"Routing accuracy:          {routing_acc:.3f}")
print(f"Balanced routing accuracy: {balanced_routing:.3f}  (chance={1/len(tasks):.3f})")
print("Per-task recall:", dict(zip(CONFIG["genres"], np.round(recall,3))))
print("Confusion matrix:"); print(cm)

print("\n=== Cross-genre generalization ===")
for j, task in enumerate(tasks):
    if task.mismatch_loader is None: continue
    e_mm,_,_ = eval_esmca(trainer, task, j, task_order, loader=task.mismatch_loader)
    print(f"{task.name:12s}  matched={esmca_accs[j]:.3f}  mismatched={e_mm:.3f}")
torch.cuda.empty_cache()

## 15. Full results + t-SNE

In [ ]:
acc=compute_acc(R); bwt=compute_bwt(R); fwt=compute_fwt(R)
ads=attribution_drift_score(trainer.drift_history); ris_score=ris(all_true,all_routed)
X,y,names = collect_sample_attributions(trainer, tasks, per_task=40)
silhouette = separability_report(X, y, names, CONFIG["output_dir"])

results = {
    "version":  "v8-llama3.2-3B-multinli-original-esmca",
    "backbone": CONFIG["backbone"],
    "dataset":  "MultiNLI (5 matched genres, NLI 3-way)",
    "config":   {k:v for k,v in CONFIG.items() if k!="output_dir"},
    "R": R, "ACC": acc, "BWT": bwt, "FWT": fwt, "ADS": ads, "RIS": ris_score,
    "routing_accuracy": routing_acc,
    "balanced_routing_accuracy": balanced_routing,
    "per_task_routing_recall": dict(zip(CONFIG["genres"], recall.tolist())),
    "confusion_matrix": cm.tolist(),
    "per_task": {t.name:{"oracle":o,"esmca":e,"uniform":u}
                 for t,o,e,u in zip(tasks,oracle_accs,esmca_accs,uniform_accs)},
    "mean_oracle":  float(np.mean(oracle_accs)),
    "mean_esmca":   float(np.mean(esmca_accs)),
    "mean_uniform": float(np.mean(uniform_accs)),
    "sample_attribution_silhouette": silhouette,
    "task_order": task_order,
}
out = os.path.join(CONFIG["output_dir"], "esmca_v8_results.json")
with open(out,"w") as f: json.dump(results,f,indent=2)
print(json.dumps({k:results[k] for k in [
    "ACC","BWT","FWT","ADS","RIS",
    "routing_accuracy","balanced_routing_accuracy",
    "mean_oracle","mean_esmca","mean_uniform",
    "sample_attribution_silhouette"]},indent=2))
print(f"Saved -> {out}")